In [0]:
from pyspark.sql import functions as F, Window

# -----------------------------
# 0) Params
# -----------------------------
dbutils.widgets.text("pde_path", "dbfs:/FileStore/pde/pde.csv")
dbutils.widgets.text("target_year", "2025")
dbutils.widgets.text("n_benes", "50000")          # synthetic beneficiaries
dbutils.widgets.text("top_drugs", "2000")         # drug pool size
dbutils.widgets.text("plan_sample_n", "300")      # number of plans to cross join per beneficiary
dbutils.widgets.text("seed", "42")

pde_path      = dbutils.widgets.get("pde_path")
target_year   = int(dbutils.widgets.get("target_year"))
n_benes       = int(dbutils.widgets.get("n_benes"))
top_drugs     = int(dbutils.widgets.get("top_drugs"))
plan_sample_n = int(dbutils.widgets.get("plan_sample_n"))
seed          = int(dbutils.widgets.get("seed"))

# spark.sql("CREATE SCHEMA IF NOT EXISTS cms_partd_synth")


# 1) Load PDE (pipe-delimited)

In [0]:

# -----------------------------
# 1) Load PDE (pipe-delimited)
# -----------------------------
pde = (
    spark.read
      .option("header", "true")
      .option("sep", "|")
      .csv(pde_path)
)

# Basic typing (keep what we need)
pde = (
    pde
    .withColumn("srvc_dt", F.to_date("SRVC_DT", "dd-MMM-yyyy"))
    .withColumn("prod_srvc_id", F.col("PROD_SRVC_ID").cast("string"))
    .withColumn("days_supply", F.col("DAYS_SUPLY_NUM").cast("int"))
    .withColumn("qty", F.col("QTY_DSPNSD_NUM").cast("double"))
    .withColumn("tot_rx_cost", F.col("TOT_RX_CST_AMT").cast("double"))
    .withColumn("ptnt_pay", F.col("PTNT_PAY_AMT").cast("double"))
    .withColumn("brand_generic", F.col("BRND_GNRC_CD").cast("string"))
    .withColumn("phrmcy_type", F.col("PHRMCY_SRVC_TYPE_CD").cast("string"))
)

# Clean obvious nulls
pde = pde.filter(F.col("prod_srvc_id").isNotNull() & F.col("tot_rx_cost").isNotNull())



# 2) Build drug statistics from PDE (for sampling + cost generation)

In [0]:
# -----------------------------
# 2) Build drug statistics from PDE (for sampling + cost generation)
# -----------------------------
drug_stats = (
    pde.groupBy("prod_srvc_id")
       .agg(
           F.count("*").alias("fills"),
           F.avg("tot_rx_cost").alias("mean_cost"),
           F.stddev("tot_rx_cost").alias("std_cost"),
           F.expr("percentile_approx(days_supply, 0.50)").alias("p50_days_supply"),
           F.avg(F.when(F.col("brand_generic") == "B", 1.0).otherwise(0.0)).alias("brand_share")
       )
)

# Keep top drugs by fills (controls generator size)
drug_top = (
    drug_stats.orderBy(F.desc("fills"))
             .limit(top_drugs)
             .withColumn("std_cost", F.when(F.col("std_cost").isNull(), F.col("mean_cost") * 0.30).otherwise(F.col("std_cost")))
            #  .cache()
)


# 3) Optional: insulin tagging for synthetic beneficiary features

In [0]:

# -----------------------------
# 3) Optional: insulin tagging for synthetic beneficiary features
#    (Join to your insulin ref if available)
# -----------------------------
insulin_ref = None
try:
    insulin_ref = spark.table("cms_partd_silver.core.slv_insulin_drug_ref")
except:
    insulin_ref = None

if insulin_ref is not None:
    # assume column name in ref is either ndc or prod_srvc_id
    ref_cols = insulin_ref.columns
    if "ndc" in ref_cols:
        insulin_keys = insulin_ref.select(F.col("ndc").cast("string").alias("prod_srvc_id")).distinct()
    elif "prod_srvc_id" in ref_cols:
        insulin_keys = insulin_ref.select(F.col("prod_srvc_id").cast("string").alias("prod_srvc_id")).distinct()
    else:
        insulin_keys = None
else:
    insulin_keys = None

if insulin_keys is not None:
    drug_top = drug_top.join(insulin_keys.withColumn("is_insulin", F.lit(1)), on="prod_srvc_id", how="left") \
                       .withColumn("is_insulin", F.coalesce(F.col("is_insulin"), F.lit(0)))
else:
    drug_top = drug_top.withColumn("is_insulin", F.lit(0))


# 4) Build a WEIGHTED sampling pool without cross-joining bene x drugs

In [0]:
# -----------------------------
# 4) Build a WEIGHTED sampling pool without cross-joining bene x drugs
#    (Approx weight by fills -> bucket -> replicate)
# -----------------------------
max_fills = drug_top.agg(F.max("fills").alias("m")).collect()[0]["m"]

drug_pool = (
    drug_top
      .withColumn("w", (F.col("fills") / F.lit(max_fills)))
      .withColumn("bucket", F.greatest(F.lit(1), F.least(F.lit(100), F.ceil(F.col("w") * 100))))  # 1..100
      .select("prod_srvc_id", "mean_cost", "std_cost", "p50_days_supply", "brand_share", "is_insulin", "bucket")
      .withColumn("rep", F.explode(F.sequence(F.lit(1), F.col("bucket"))))
      .drop("rep")
)

# Create an index for uniform sampling over replicated rows
w = Window.orderBy(F.monotonically_increasing_id())
drug_pool = drug_pool.withColumn("pool_idx", F.row_number().over(w))

pool_n = drug_pool.count()


# 5) Generate synthetic beneficiaries (risk segments + insulin user flag)

In [0]:
# -----------------------------
# 5) Generate synthetic beneficiaries (risk segments + GEOGRAPHY)
# -----------------------------
bene = spark.range(0, n_benes).withColumnRenamed("id", "bene_num")

# Load Geography Distribution (from Real Data)
dim_geo = spark.table("cms_partd_gold.mart.dim_geography")

# Assign State/County based on population or random uniform from dim_geo
# Optimization: Broadcast a list of valid county_codes and sample
geo_sample = dim_geo.select("state", "county_code").sample(withReplacement=True, fraction=1.0).limit(n_benes)
# Note: This is an approximation. Ideally weighted by population.
geo_sample = geo_sample.withColumn("bene_num", F.monotonically_increasing_id())
geo_sample = geo_sample.withColumn("bene_num", F.col("bene_num") % n_benes) # Wrap around if limits mismatch

bene = bene.join(geo_sample, on="bene_num", how="left")

# Load Zipcodes for that County
dim_zip = spark.table("cms_partd_gold.mart.dim_zipcode_geo")

# Assign Random ZipCode within that County
# Simple Strategy: Join with dim_zip on county_code, then pick one random per bene
# (Efficiency Warning: A full join is expensive. We will do a Window over Partition approximation)
window_spec = Window.partitionBy("county_code").orderBy(F.rand())
full_bene_geo = bene.join(dim_zip.select("county_code", "zip_code"), on="county_code", how="inner") \
    .withColumn("rank", F.row_number().over(window_spec)) \
    .filter(F.col("rank") == 1) \
    .drop("rank")

bene = full_bene_geo

# Risk segments: low/med/high (tune these shares)
bene = (
    bene
    .withColumn("u", F.rand(seed))
    .withColumn(
        "risk_segment",
        F.when(F.col("u") < 0.65, F.lit("LOW"))
         .when(F.col("u") < 0.90, F.lit("MED"))
         .otherwise(F.lit("HIGH"))
    )
)

# Segment-specific expected unique drugs + fills (simple + controllable)
bene = (
    bene
    .withColumn("exp_unique_drugs",
        F.when(F.col("risk_segment")=="LOW",  F.lit(3))
         .when(F.col("risk_segment")=="MED",  F.lit(7))
         .otherwise(F.lit(14))
    )
    .withColumn("exp_fills",
        F.when(F.col("risk_segment")=="LOW",  F.lit(8))
         .when(F.col("risk_segment")=="MED",  F.lit(18))
         .otherwise(F.lit(40))
    )
)

# Normal approximation to Poisson: round(lambda + N(0, sqrt(lambda)))
bene = (
    bene
    .withColumn("unique_drugs",
        F.greatest(F.lit(1), F.least(F.lit(30),
            F.round(F.col("exp_unique_drugs") + F.randn(seed+1) * F.sqrt(F.col("exp_unique_drugs")))
        )).cast("int")
    )
    .withColumn("fills_target",
        F.greatest(F.lit(1), F.least(F.lit(120),
            F.round(F.col("exp_fills") + F.randn(seed+2) * F.sqrt(F.col("exp_fills")))
        )).cast("int")
    )
)

# Insulin user probability higher for MED/HIGH
bene = (
    bene
    .withColumn("insulin_user_flag",
        F.when((F.col("risk_segment")=="HIGH") & (F.rand(seed+3) < 0.35), 1)
         .when((F.col("risk_segment")=="MED")  & (F.rand(seed+4) < 0.18), 1)
         .when((F.col("risk_segment")=="LOW")  & (F.rand(seed+5) < 0.05), 1)
         .otherwise(0)
    )
)

# Channel preference (retail vs mail)
bene = (
    bene
    .withColumn("channel_pref",
        F.when(F.rand(seed+6) < 0.75, F.lit("RETAIL")).otherwise(F.lit("MAIL"))
    )
)

# Create stable synthetic id
bene = bene.withColumn("bene_synth_id", F.sha2(F.concat_ws(":", F.lit("BENE"), F.col("bene_num").cast("string")), 256))

# Save beneficiary table
(bene.select("bene_synth_id","risk_segment","unique_drugs","fills_target","insulin_user_flag","channel_pref", "state", "county_code", "zip_code")
     .write.format("delta").mode("overwrite").saveAsTable("cms_partd_synth.synth_data.syn_beneficiary"))


# 6) Assign drug baskets to beneficiaries (uniform over weighted pool)

In [0]:
# -----------------------------
# 6) Assign drug baskets to beneficiaries (uniform over weighted pool)
#    We generate unique_drugs rows per beneficiary, each picks a pool_idx
# -----------------------------
bene_drug = (
    bene.select("bene_synth_id","unique_drugs","fills_target","insulin_user_flag")
        .withColumn("k", F.explode(F.sequence(F.lit(1), F.col("unique_drugs"))))
        .withColumn("pool_idx", (F.floor(F.rand(seed+10) * F.lit(pool_n)) + 1).cast("int"))
        .join(drug_pool, on="pool_idx", how="left")
        .drop("pool_idx","k")
)

# Ensure insulin users have at least 1 insulin drug if insulin list exists in pool
if drug_pool.filter(F.col("is_insulin")==1).limit(1).count() > 0:
    # Force at least one insulin drug row for insulin users by replacing the first row with an insulin drug
    insulin_one = drug_pool.filter(F.col("is_insulin")==1).orderBy(F.rand(seed+11)).limit(1).select(
        "prod_srvc_id","mean_cost","std_cost","p50_days_supply","brand_share","is_insulin"
    ).collect()[0]
    bene_drug = bene_drug.withColumn(
        "force_insulin",
        F.when(F.col("insulin_user_flag")==1, F.row_number().over(Window.partitionBy("bene_synth_id").orderBy(F.rand(seed+12)))).otherwise(F.lit(999999))
    )

    bene_drug = (
        bene_drug
        .withColumn("prod_srvc_id",
            F.when(F.col("force_insulin")==1, F.lit(insulin_one["prod_srvc_id"])).otherwise(F.col("prod_srvc_id"))
        )
        .withColumn("mean_cost",
            F.when(F.col("force_insulin")==1, F.lit(float(insulin_one["mean_cost"]))).otherwise(F.col("mean_cost"))
        )
        .withColumn("std_cost",
            F.when(F.col("force_insulin")==1, F.lit(float(insulin_one["std_cost"]))).otherwise(F.col("std_cost"))
        )
        .withColumn("p50_days_supply",
            F.when(F.col("force_insulin")==1, F.lit(int(insulin_one["p50_days_supply"]))).otherwise(F.col("p50_days_supply"))
        )
        .withColumn("is_insulin",
            F.when(F.col("force_insulin")==1, F.lit(int(insulin_one["is_insulin"]))).otherwise(F.col("is_insulin"))
        )
        .drop("force_insulin")
    )


# 7) Generate synthetic PDE-like events for target_year

In [0]:

# -----------------------------
# 7) Generate synthetic PDE-like events for target_year
# -----------------------------
start_date = F.to_date(F.lit(f"{target_year}-01-01"))
days_in_year = 365

# Allocate fills across drugs (roughly proportional): each drug gets at least 1 fill
# Here we assign a per-drug fill count based on beneficiary fills_target / unique_drugs with randomness
bene_drug = (
    bene_drug
    .withColumn("lambda_drug",
        F.greatest(F.lit(0.5), (F.col("fills_target") / F.col("unique_drugs")) * (0.5 + F.rand(seed+20)))
    )
    .withColumn("fills_drug",
        F.greatest(F.lit(1), F.round(F.col("lambda_drug") + F.randn(seed+21) * F.sqrt(F.col("lambda_drug")))).cast("int")
    )
)

events = (
    bene_drug
    .withColumn("fill_seq", F.explode(F.sequence(F.lit(1), F.col("fills_drug"))))
    .withColumn("srvc_dt", F.date_add(start_date, F.floor(F.rand(seed+30) * F.lit(days_in_year)).cast("int")))
    .withColumn("days_supply", F.coalesce(F.col("p50_days_supply").cast("int"), F.lit(30)))
    .withColumn("qty", (10 + F.rand(seed+31)*90).cast("double"))
)

# Generate costs from mean/std (normal, clipped >0)
events = (
    events
    .withColumn("tot_rx_cost", F.greatest(F.lit(1.0), F.col("mean_cost") + F.col("std_cost") * F.randn(seed+40)))
    .withColumn("brand_generic", F.when(F.rand(seed+41) < F.col("brand_share"), F.lit("B")).otherwise(F.lit("G")))
)

# Keep only essential event fields
syn_pde = events.select(
    "bene_synth_id","srvc_dt","prod_srvc_id","is_insulin",
    "days_supply","qty","tot_rx_cost","brand_generic"
)

(syn_pde.write.format("delta").mode("overwrite").option("mergeSchema", "True").saveAsTable("cms_partd_synth.synth_data.syn_pde_events"))


# 8) Beneficiary-year summary (beneficiary features)

In [0]:

# -----------------------------
# 8) Beneficiary-year summary (beneficiary features)
# -----------------------------
bene_year = (
    syn_pde
    .groupBy("bene_synth_id")
    .agg(
        F.count("*").alias("fills"),
        F.countDistinct("prod_srvc_id").alias("unique_drugs"),
        F.sum("tot_rx_cost").alias("total_rx_cost"),
        F.sum(F.when(F.col("is_insulin")==1, 1).otherwise(0)).alias("insulin_fills"),
        F.sum(F.when(F.col("is_insulin")==1, F.col("tot_rx_cost")).otherwise(0.0)).alias("insulin_rx_cost"),
    )
    .withColumn("insulin_cost_share",
        F.when(F.col("total_rx_cost") > 0, F.col("insulin_rx_cost") / F.col("total_rx_cost")).otherwise(F.lit(0.0))
    )
)

# (bene_year.write.format("delta").mode("overwrite").saveAsTable("cms_partd_synth.synth_data.syn_bene_year_summary"))


# 9) Create training pairs for Model 1 & Model 2

In [0]:
# -----------------------------
# 9) Create training pairs: Service Area Enforced
# -----------------------------
plans_all = spark.table("cms_partd_gold.ml.training_plan_labels")
plans_all = plans_all.filter((F.col("plan_suppressed_yn").isNull()) | (F.col("plan_suppressed_yn") != "Y"))

drop_cols = ["premium_z","deductible_z","pref_avg_copay_amt_z","pref_p90_copay_amt_z", "pref_avg_coins_rate_z","pref_p90_coins_rate_z","ded_applies_rate_z", "affordability_index"]
plans_all = plans_all.drop(*[c for c in drop_cols if c in plans_all.columns])

# JOIN condition: Match Plan Service Area to Bene Location
# Plans are either National/Regional (PDP) or County-Specific (MA-PD, etc)
# We join on State + CountyCode approx. 
# Note: dim_plan has county_code. If Plan serves a region, it might have multiple rows or a Region code.
# Assuming feature_plan_summary (and training_plan_labels) is built from dim_plan which IS county-grain.

pairs = bene_year.join(bene.select("bene_synth_id", "state", "county_code"), on="bene_synth_id") \
    .join(plans_all, on=["state", "county_code"], how="inner")

# Downsample if too huge (Stratified per bene?)
pairs = pairs.withColumn("rand", F.rand()).filter(F.col("rand") < 0.1).drop("rand")


In [0]:
# -----------------------------
# 10) Label engineering: synthetic annual OOP (regression target)
#     This uses your cost structure metrics correctly:
#     - copay ($) vs coinsurance (rate)
# -----------------------------
# annual premium
pairs = pairs.withColumn("annual_premium", F.coalesce(F.col("premium"), F.lit(0.0)) * 12.0)

# expected OOP from copay component ($): fills * share * avg_copay
pairs = pairs.withColumn(
    "oop_copay_component",
    F.col("fills") * F.coalesce(F.col("pref_copay_share"), F.lit(0.0)) * F.coalesce(F.col("pref_avg_copay_amt"), F.lit(0.0))
)

# expected OOP from coinsurance component ($): total_cost * share * avg_coins_rate
pairs = pairs.withColumn(
    "oop_coins_component",
    F.col("total_rx_cost") * F.coalesce(F.col("pref_coinsurance_share"), F.lit(0.0)) * F.coalesce(F.col("pref_avg_coins_rate"), F.lit(0.0))
)

# deductible approximation: deductible applies rate * deductible capped by total cost
pairs = pairs.withColumn(
    "oop_deductible_component",
    F.coalesce(F.col("ded_applies_rate"), F.lit(0.0)) * F.least(F.coalesce(F.col("deductible"), F.lit(0.0)), F.col("total_rx_cost"))
)

# insulin cap approximation (if insulin_user): cap insulin-related OOP at $35*12 when plan is "insulin friendly"
# You can tune this logic to use your insulin compliance flags/scores.
pairs = pairs.withColumn("insulin_user_flag", F.when(F.col("insulin_fills") > 0, 1).otherwise(0))

pairs = pairs.withColumn(
    "oop_insulin_raw",
    (F.col("insulin_rx_cost") * F.coalesce(F.col("pref_avg_coins_rate"), F.lit(0.0))) +
    (F.col("insulin_fills") * F.coalesce(F.col("pref_avg_copay_amt"), F.lit(0.0)) * 0.1)  # small copay proxy
)

pairs = pairs.withColumn(
    "oop_insulin_capped",
    F.when(
        (F.col("insulin_user_flag")==1) & (F.coalesce(F.col("insulin_risk_flag"), F.lit(0))==0),
        F.least(F.col("oop_insulin_raw"), F.lit(35.0*12.0))
    ).otherwise(F.col("oop_insulin_raw"))
)

# Add penalty if insulin access risk is high (prior auth / restrictive)
pairs = pairs.withColumn(
    "insulin_access_penalty",
    F.when((F.col("insulin_user_flag")==1) & (F.coalesce(F.col("insulin_access_risk_flag"), F.lit(0))==1),
           F.lit(150.0)  # tune: admin burden / extra costs proxy
    ).otherwise(F.lit(0.0))
)

# Network penalty (travel/time burden proxy)
pairs = pairs.withColumn(
    "network_penalty",
    F.when(F.coalesce(F.col("network_adequacy_flag"), F.lit(0))==1, F.lit(100.0)).otherwise(F.lit(0.0))
)

# Final annual OOP label
pairs = pairs.withColumn(
    "y_annual_oop",
    F.col("annual_premium")
    + F.col("oop_deductible_component")
    + F.col("oop_copay_component")
    + F.col("oop_coins_component")
    - F.col("oop_insulin_raw") + F.col("oop_insulin_capped")
    + F.col("insulin_access_penalty")
    + F.col("network_penalty")
)



In [0]:
# Retrieve risk_segment
pairs = pairs.join(spark.sql("select bene_synth_id, risk_segment from cms_partd_synth.synth_data.syn_beneficiary"), how="left", on="bene_synth_id")

In [0]:

# -----------------------------
# 11) Suitability labels (classification targets)
#     Example: 3-class suitability based on OOP + insulin/network risk
# -----------------------------
pairs = pairs.withColumn("network_bad", F.when(F.col("network_adequacy_flag")==1, 1).otherwise(0))
pairs = pairs.withColumn("insulin_bad", F.when(F.col("insulin_access_risk_flag")==1, 1).otherwise(0))

# Thresholds based on beneficiary cost distribution (quantiles)
q1, q2 = pairs.approxQuantile("y_annual_oop", [0.33, 0.66], 0.01)

pairs = pairs.withColumn("oop_band",
    F.when(F.col("y_annual_oop") <= F.lit(q1), F.lit(0))
     .when(F.col("y_annual_oop") <= F.lit(q2), F.lit(1))
     .otherwise(F.lit(2))
)

# Hard warnings
pairs = pairs.withColumn("warn_network", F.when(F.coalesce(F.col("network_adequacy_flag"), F.lit(0))==1, 1).otherwise(0))
pairs = pairs.withColumn("warn_insulin_cap",
    F.when((F.col("insulin_user_flag")==1) & (F.coalesce(F.col("insulin_cap_violation_flag"), F.lit(0))==1), 1).otherwise(0)
)
pairs = pairs.withColumn("warn_insulin_access",
    F.when((F.col("insulin_user_flag")==1) & (F.coalesce(F.col("insulin_access_risk_flag"), F.lit(0))==1), 1).otherwise(0)
)

# Restrictiveness matters more for higher-utilization beneficiaries
# Example: if beneficiary is HIGH utilization and plan restrictiveness_class=2, count as warning
pairs = pairs.withColumn(
    "warn_restrict",
    F.when((F.col("risk_segment")=="HIGH") & (F.coalesce(F.col("restrictiveness_class"), F.lit(1))==2), 1).otherwise(0)
)

pairs = pairs.withColumn(
    "warning_score",
    F.col("warn_network") + F.col("warn_insulin_cap") + F.col("warn_insulin_access") + F.col("warn_restrict")
)

# Suitability class
pairs = pairs.withColumn(
    "suitability_class",
    F.when((F.col("oop_band")==0) & (F.col("warning_score")==0), 0)
     .when((F.col("oop_band")==2) | (F.col("warning_score") >= 2), 2)
     .otherwise(1)
)

In [0]:
pairs = pairs.withColumn(
    "explain_network",
    F.when(F.col("warn_network")==1, F.lit("Limited network coverage in your area")).otherwise(F.lit(None))
)

pairs = pairs.withColumn(
    "explain_insulin_cap",
    F.when(F.col("warn_insulin_cap")==1, F.lit("Insulin cost cap may not apply / potential violation")).otherwise(F.lit(None))
)

pairs = pairs.withColumn(
    "explain_insulin_access",
    F.when(F.col("warn_insulin_access")==1, F.lit("Insulin access may require restrictions (PA/ST/QL)")).otherwise(F.lit(None))
)

pairs = pairs.withColumn(
    "explain_restrict",
    F.when(F.col("warn_restrict")==1, F.lit("High formulary restrictiveness for high-utilization profile")).otherwise(F.lit(None))
)


In [0]:
(pairs.write.format("delta")
      .mode("overwrite")
      .option("overwriteSchema","true")
      .saveAsTable("cms_partd_synth.synth_data.training_plan_pairs"))


# Diagnostics

In [0]:
from pyspark.sql import functions as F

pairs = spark.table("cms_partd_synth.synth_data.training_plan_pairs")

pairs.groupBy("oop_band").count().orderBy("oop_band").show()
pairs.groupBy("warning_score").count().orderBy("warning_score").show()

pairs.filter(F.col("oop_band")==0).groupBy("warning_score").count().orderBy("warning_score").show()

pairs.groupBy("network_adequacy_flag").count().show()
pairs.groupBy("risk_segment","restrictiveness_class").count().show()
pairs.groupBy("insulin_user_flag","insulin_cap_violation_flag","insulin_access_risk_flag").count().show()
